# Frozen NVARC hidden states → FLUX.2 Klein Base 4B

This Colab notebook runs one diagnostic inference path from a frozen NVARC checkpoint into the existing FLUX.2 text projection and decodes the result with the standalone FLUX.2 VAE.

> **Interpretation:** FLUX has not been trained on NVARC hidden states. A technically successful run may therefore produce an invalid or chaotic ARC canvas. This is an interface experiment, not evidence of ARC-solving quality.

## Before running

1. In Colab, select an A100 GPU runtime.
2. Confirm that the three checkpoints are present under MyDrive/diffusion-arc/models.
3. In the Colab Secrets panel, create GITHUB_TOKEN and allow this notebook to access it. Use a fine-grained, read-only token limited to the private diffusion-arc repository with Contents: Read permission.
4. Run the notebook from top to bottom. No generated image or token is saved.

## 1. Mount Drive and verify the A100 runtime

In [ ]:
from pathlib import Path
import platform

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime in Colab first.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This experiment requires a CUDA GPU with BF16 support.")

gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 2**30
if gpu_gib < 38:
    raise RuntimeError(
        f"The selected GPU exposes only {gpu_gib:.1f} GiB. Use an A100 40 GB or larger."
    )

DRIVE_MOUNT = Path("/content/drive")
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
if not MY_DRIVE.is_dir():
    drive.mount(str(DRIVE_MOUNT), force_remount=True)
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive was mounted, but MyDrive is not accessible.")

print(f"Python: {platform.python_version()}")
print(f"Torch: {torch.__version__} | CUDA runtime: {torch.version.cuda}")
print(f"GPU: {gpu.name} | VRAM: {gpu_gib:.1f} GiB | BF16: yes")
print(f"Drive: {MY_DRIVE}")

## 2. Fetch the private repository at the pinned commit

The token is read from Colab Secrets inside a function. It is never placed in a URL, environment variable, shell command, notebook output, or Drive. The archive redirect is accepted only when GitHub points to its official codeload host.

In [ ]:
import hashlib
import re
import shutil
import stat
import tempfile
import zipfile
from pathlib import PurePosixPath
from urllib.parse import urlparse

import requests
from google.colab import userdata

GITHUB_OWNER = "joshuakauling"
GITHUB_REPOSITORY = "diffusion-arc"
SOURCE_COMMIT = "5ac06150fb05c5a87dbc28654e9e4e682e3bc98e"
SOURCE_DIR = Path(f"/content/diffusion-arc-{SOURCE_COMMIT[:12]}")
MAX_ARCHIVE_BYTES = 200 * 1024 * 1024


def _require_safe_zip(archive_path: Path, destination: Path) -> str:
    if not zipfile.is_zipfile(archive_path):
        raise RuntimeError("GitHub did not return a valid ZIP archive.")
    destination_root = destination.resolve()
    top_levels = set()
    with zipfile.ZipFile(archive_path) as archive:
        corrupt_member = archive.testzip()
        if corrupt_member is not None:
            raise RuntimeError("The downloaded source archive is corrupt.")
        for info in archive.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or ".." in member.parts or not member.parts:
                raise RuntimeError("The source archive contains an unsafe path.")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError("The source archive contains a symbolic link.")
            target = (destination_root / Path(*member.parts)).resolve()
            try:
                target.relative_to(destination_root)
            except ValueError as error:
                raise RuntimeError("The source archive escapes its extraction directory.") from error
            top_levels.add(member.parts[0])
        if len(top_levels) != 1:
            raise RuntimeError("Expected exactly one top-level directory in the source archive.")
        archive.extractall(destination_root)
    return next(iter(top_levels))


def prepare_private_source() -> tuple[Path, str]:
    marker = SOURCE_DIR / ".source-commit"
    required = SOURCE_DIR / "src" / "diffusion_arc" / "model.py"
    if SOURCE_DIR.is_dir():
        if required.is_file() and marker.is_file() and marker.read_text().strip() == SOURCE_COMMIT:
            return SOURCE_DIR, "reused"
        raise RuntimeError(
            f"{SOURCE_DIR} already exists but is incomplete or unverified. "
            "Reconnect the disposable Colab runtime and rerun the notebook."
        )

    if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
        raise RuntimeError("SOURCE_COMMIT must be a full 40-character Git commit SHA.")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Colab Secret GITHUB_TOKEN is missing or notebook access is disabled."
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("Colab Secret GITHUB_TOKEN is empty.")
    token = token.strip()

    api_root = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPOSITORY}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    verification = requests.get(
        f"{api_root}/commits/{SOURCE_COMMIT}", headers=headers, timeout=(10, 60)
    )
    verification_status = verification.status_code
    if verification_status != 200:
        verification.close()
        raise RuntimeError(
            f"GitHub commit verification failed with HTTP {verification_status}. "
            "Check the secret and its repository permissions."
        )
    resolved_commit = verification.json().get("sha")
    verification.close()
    if resolved_commit != SOURCE_COMMIT:
        raise RuntimeError("GitHub did not resolve the exact pinned commit.")

    initial = requests.get(
        f"{api_root}/zipball/{SOURCE_COMMIT}",
        headers=headers,
        stream=True,
        allow_redirects=False,
        timeout=(10, 120),
    )
    del token, headers

    if initial.status_code in {301, 302, 303, 307, 308}:
        location = initial.headers.get("Location", "")
        initial.close()
        parsed = urlparse(location)
        if parsed.scheme != "https" or parsed.hostname != "codeload.github.com":
            raise RuntimeError("GitHub returned an unexpected archive redirect host.")
        response = requests.get(
            location, stream=True, allow_redirects=False, timeout=(10, 120)
        )
        del location
    else:
        response = initial

    if response.status_code != 200:
        status = response.status_code
        response.close()
        raise RuntimeError(f"GitHub source download failed with HTTP {status}.")

    SOURCE_DIR.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="diffusion-arc-source-", dir="/content") as temp:
        temp_dir = Path(temp)
        partial_archive = temp_dir / "source.zip.part"
        archive_path = temp_dir / "source.zip"
        digest = hashlib.sha256()
        downloaded = 0
        with partial_archive.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                downloaded += len(chunk)
                if downloaded > MAX_ARCHIVE_BYTES:
                    raise RuntimeError("The source archive exceeded the 200 MiB safety limit.")
                digest.update(chunk)
                handle.write(chunk)
        response.close()
        partial_archive.replace(archive_path)

        extraction_dir = temp_dir / "extracted"
        extraction_dir.mkdir()
        top_level = _require_safe_zip(archive_path, extraction_dir)
        extracted_source = extraction_dir / top_level
        if not (extracted_source / "pyproject.toml").is_file():
            raise RuntimeError("The archive does not contain the expected Python project.")
        shutil.move(str(extracted_source), str(SOURCE_DIR))
        (SOURCE_DIR / ".source-commit").write_text(SOURCE_COMMIT + "\n")
        (SOURCE_DIR / ".source-archive-sha256").write_text(digest.hexdigest() + "\n")

    return SOURCE_DIR, "downloaded"


SOURCE_DIR, source_status = prepare_private_source()
print(f"Private source: {source_status} at commit {SOURCE_COMMIT[:12]}")
print(f"Runtime path: {SOURCE_DIR}")

## 3. Install the minimal pinned model dependencies

This intentionally does not install requirements.txt and does not replace Colab's CUDA-enabled Torch build. Installation output is captured and only shown if a command fails.

In [ ]:
import importlib.metadata
import subprocess
import sys

PINNED_PACKAGES = (
    "accelerate==1.12.0",
    "einops==0.8.1",
    "safetensors==0.8.0",
    "transformers==4.56.1",
)


def run_quiet(label: str, command: list[str]) -> None:
    completed = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = "\n".join(completed.stdout.splitlines()[-40:])
        print(tail)
        raise RuntimeError(f"{label} failed with exit code {completed.returncode}.")
    print(f"{label}: ok")


run_quiet(
    "Pinned dependencies",
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-input",
        *PINNED_PACKAGES,
    ],
)
run_quiet(
    "Diffusion-ARC source",
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-input",
        "--no-deps",
        "--no-build-isolation",
        "-e",
        str(SOURCE_DIR),
    ],
)

import diffusion_arc

package_path = Path(diffusion_arc.__file__).resolve()
if not package_path.is_relative_to((SOURCE_DIR / "src").resolve()):
    raise RuntimeError(f"diffusion_arc resolved from an unexpected path: {package_path}")

version_names = (
    "torch",
    "transformers",
    "accelerate",
    "safetensors",
    "einops",
    "tokenizers",
    "huggingface-hub",
)
versions = {name: importlib.metadata.version(name) for name in version_names}
print("Versions:", versions)
print("Note: Colab Torch is intentionally retained even though pyproject.toml pins Torch 2.13.0.")

## 4. Configure and validate the local checkpoints

The default skips re-hashing the multi-gigabyte FLUX and VAE files because the download notebook already verified them. Set VERIFY_CHECKPOINT_HASHES to True when you explicitly want a slower identity check.

In [ ]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

DEVICE = torch.device("cuda:0")
NVARC_DTYPE = torch.bfloat16
FLUX_DTYPE = torch.bfloat16
VAE_DTYPE = torch.float32
NVARC_LAYERS = (9, 18, 27)
VERIFY_CHECKPOINT_HASHES = False

MODEL_ROOT = MY_DRIVE / "diffusion-arc" / "models"
NVARC_PATH = MODEL_ROOT / "nvarc-qwen3-4b-grids15-sft139"
FLUX_PATH = MODEL_ROOT / "flux2-klein-base-4b" / "flux-2-klein-base-4b.safetensors"
VAE_PATH = MODEL_ROOT / "flux2-standalone-vae" / "ae.safetensors"

required_nvarc_files = (
    "config.json",
    "model-00001-of-00002.safetensors",
    "model-00002-of-00002.safetensors",
    "model.safetensors.index.json",
    "tokenizer.json",
)
missing_nvarc = [name for name in required_nvarc_files if not (NVARC_PATH / name).is_file()]
if missing_nvarc:
    raise FileNotFoundError(f"Incomplete NVARC checkpoint: {missing_nvarc}")
for checkpoint in (FLUX_PATH, VAE_PATH):
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint}")

print(f"NVARC directory: {NVARC_PATH}")
print(f"FLUX: {FLUX_PATH} ({FLUX_PATH.stat().st_size / 2**30:.2f} GiB)")
print(f"VAE: {VAE_PATH} ({VAE_PATH.stat().st_size / 2**30:.2f} GiB)")
print(f"Checkpoint re-hashing: {VERIFY_CHECKPOINT_HASHES}")

## 5. Load NVARC, FLUX, and the standalone VAE

All components are loaded from Drive with local-files-only behavior. NVARC and the VAE are frozen by the project model. FLUX remains structurally trainable, but this notebook uses evaluation and inference mode only—there is no optimizer, backward pass, or weight update.

In [ ]:
import gc
import time

from diffusion_arc.loading import (
    OFFICIAL_FLUX2_KLEIN_BASE_4B_SHA256,
    OFFICIAL_FLUX2_VAE_SHA256,
    load_bfl_autoencoder_local,
    load_bfl_flux2_klein_4b_local,
    load_nvarc_local,
)
from diffusion_arc.model import (
    ARC_IMAGE_TOKENS,
    ARC_LATENT_CHANNELS,
    ARC_LATENT_HEIGHT,
    ARC_LATENT_WIDTH,
    DiffusionARCGenerationOutput,
    DiffusionARCModel,
)
from diffusion_arc.nvarc_conditioning import NVARCConditioner
from diffusion_arc.positions import make_image_ids, make_text_ids
from diffusion_arc.sampling import bfl_time_schedule, euler_integrate

stale_gpu_names = (
    "generation",
    "raw_image",
    "decode",
    "context",
    "model",
    "conditioner",
    "nvarc",
    "flux",
    "vae",
    "tokenizer",
)
for stale_name in stale_gpu_names:
    if stale_name in globals():
        del globals()[stale_name]
gc.collect()
torch.cuda.empty_cache()
load_started = time.perf_counter()

nvarc, tokenizer, nvarc_contract = load_nvarc_local(
    NVARC_PATH,
    device=DEVICE,
    dtype=NVARC_DTYPE,
)
print("NVARC loaded and its tokenizer/model contract passed.")

flux = load_bfl_flux2_klein_4b_local(
    FLUX_PATH,
    device=DEVICE,
    dtype=FLUX_DTYPE,
    expected_sha256=(
        OFFICIAL_FLUX2_KLEIN_BASE_4B_SHA256 if VERIFY_CHECKPOINT_HASHES else None
    ),
)
print("FLUX transformer loaded.")

vae = load_bfl_autoencoder_local(
    VAE_PATH,
    device=DEVICE,
    dtype=VAE_DTYPE,
    expected_sha256=OFFICIAL_FLUX2_VAE_SHA256 if VERIFY_CHECKPOINT_HASHES else None,
)
print("Standalone VAE loaded.")

conditioner = NVARCConditioner(nvarc, layer_indices=NVARC_LAYERS)
model = DiffusionARCModel(conditioner=conditioner, flux=flux, vae=vae).eval()
torch.cuda.synchronize(DEVICE)

if any(parameter.requires_grad for parameter in model.nvarc.parameters()):
    raise RuntimeError("NVARC is not fully frozen.")
if any(parameter.requires_grad for parameter in model.vae.parameters()):
    raise RuntimeError("The VAE is not fully frozen.")
if model.flux.txt_in.in_features != 7680:
    raise RuntimeError("FLUX does not expose the required 7,680-wide text input.")

gc.collect()
torch.cuda.empty_cache()
print(f"Load time: {time.perf_counter() - load_started:.1f} s")
print(
    "NVARC contract:",
    {
        "hidden_size": nvarc_contract["hidden_size"],
        "layers": nvarc_contract["num_hidden_layers"],
        "tokenizer_size": nvarc_contract["tokenizer_size"],
    },
)
print(f"NVARC hidden-state indices: {NVARC_LAYERS} → 3 × 2,560 = 7,680")
print(f"CUDA allocated after load: {torch.cuda.memory_allocated(DEVICE) / 2**30:.2f} GiB")

## 6. Define a small ARC task

Edit TASK to try another problem. Keep EXPECTED_GRID separate: it is used only for evaluation after generation and is never passed to NVARC or FLUX.

In [ ]:
TASK = {
    "train": [
        {
            "input": [[2, 0, 0], [0, 0, 0], [0, 0, 0]],
            "output": [[0, 0, 0], [0, 2, 0], [0, 0, 0]],
        },
        {
            "input": [[0, 0, 0], [0, 0, 0], [0, 0, 2]],
            "output": [[0, 0, 0], [0, 2, 0], [0, 0, 0]],
        },
    ],
    "test": [
        {"input": [[0, 0, 2], [0, 0, 0], [0, 0, 0]]},
    ],
}
TEST_INDEX = 0
EXPECTED_GRID = [[0, 0, 0], [0, 2, 0], [0, 0, 0]]  # Or set to None.

print("Task ready. Edit TASK and EXPECTED_GRID above to try another problem.")

## 7. Serialize, tokenize, and extract the NVARC conditioning once

In [ ]:
import copy
import hashlib

from diffusion_arc.serialization import serialize_arc_task

task_for_prompt = copy.deepcopy(TASK)
for test_example in task_for_prompt["test"]:
    test_example.pop("output", None)
if any("output" in example for example in task_for_prompt["test"]):
    raise AssertionError("A test answer leaked into the NVARC task.")

prompt = serialize_arc_task(task_for_prompt, test_index=TEST_INDEX)
if not prompt.endswith("<|im_start|>assistant\n"):
    raise RuntimeError("The serialized NVARC prompt has an unexpected suffix.")

encoded = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
    padding=False,
    truncation=False,
)
input_ids = encoded["input_ids"]
attention_mask = encoded.get("attention_mask", torch.ones_like(input_ids))
if input_ids.shape != attention_mask.shape or input_ids.shape[0] != 1:
    raise RuntimeError("NVARC requires one exact, unpadded token sequence.")
if not 1 <= input_ids.shape[1] <= 8192:
    raise RuntimeError(f"Prompt length is outside [1, 8192]: {input_ids.shape[1]}")
if not bool((attention_mask == 1).all().item()):
    raise RuntimeError("The NVARC prompt was unexpectedly padded or masked.")

prompt_sha256 = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
print(f"Prompt tokens: {input_ids.shape[1]} | SHA-256: {prompt_sha256}")
print("Test output included in conditioning: no")
print("\nSerialized NVARC prompt:\n")
print(prompt)

In [ ]:
context_started = time.perf_counter()
with torch.inference_mode():
    context = model.encode_conditioning(input_ids, attention_mask)
torch.cuda.synchronize(DEVICE)

expected_context_shape = (1, input_ids.shape[1], 7680)
if tuple(context.shape) != expected_context_shape:
    raise RuntimeError(
        f"Conditioning shape is {tuple(context.shape)}, expected {expected_context_shape}."
    )
if context.dtype != FLUX_DTYPE or context.device != DEVICE:
    raise RuntimeError("Conditioning does not match the FLUX device and BF16 dtype.")

context_float = context.float()
context_stats = {
    "shape": tuple(context.shape),
    "dtype": str(context.dtype),
    "device": str(context.device),
    "mean": float(context_float.mean().item()),
    "std": float(context_float.std().item()),
    "min": float(context_float.min().item()),
    "max": float(context_float.max().item()),
    "finite": bool(torch.isfinite(context).all().item()),
    "seconds": time.perf_counter() - context_started,
}
del context_float
gc.collect()
torch.cuda.empty_cache()
print("Direct NVARC → FLUX context:", context_stats)

## 8. Sample from FLUX using the cached NVARC context

Changing only SEED or NUM_STEPS requires rerunning this cell and the display cell—not NVARC. A single progress bar replaces per-step printing.

In [ ]:
from tqdm.auto import tqdm


@torch.inference_mode()
def generate_from_context(
    model: DiffusionARCModel,
    cached_context: torch.Tensor,
    *,
    num_steps: int,
    seed: int,
) -> DiffusionARCGenerationOutput:
    if isinstance(num_steps, bool) or not isinstance(num_steps, int) or num_steps <= 0:
        raise ValueError("num_steps must be a positive integer.")
    flux_parameter = next(
        parameter
        for parameter in model.flux.parameters()
        if torch.is_floating_point(parameter)
    )
    flux_device = flux_parameter.device
    flux_dtype = flux_parameter.dtype
    batch = cached_context.shape[0]
    if cached_context.device != flux_device or cached_context.dtype != flux_dtype:
        raise RuntimeError("Cached context does not match FLUX device and dtype.")

    generator = torch.Generator(device=flux_device).manual_seed(seed)
    latents = torch.randn(
        (batch, ARC_IMAGE_TOKENS, ARC_LATENT_CHANNELS),
        device=flux_device,
        dtype=flux_dtype,
        generator=generator,
    )
    image_ids = make_image_ids(
        batch,
        ARC_LATENT_HEIGHT,
        ARC_LATENT_WIDTH,
        device=flux_device,
    )
    text_ids = make_text_ids(batch, cached_context.shape[1], device=flux_device)
    schedule = bfl_time_schedule(
        num_steps,
        ARC_IMAGE_TOKENS,
        device=flux_device,
        dtype=torch.float32,
    )

    progress = tqdm(total=num_steps, desc="FLUX denoising", unit="step", mininterval=0.5)

    def velocity_fn(state: torch.Tensor, time_batch: torch.Tensor) -> torch.Tensor:
        velocity = model.predict_velocity(
            latents=state,
            timesteps=time_batch,
            context=cached_context,
            image_ids=image_ids,
            text_ids=text_ids,
        )
        progress.update(1)
        return velocity

    try:
        latents = euler_integrate(latents, velocity_fn, schedule)
    finally:
        progress.close()
    images, decodes = model.decode_latents(latents)
    return DiffusionARCGenerationOutput(
        latents=latents,
        images=images,
        decodes=decodes,
        timesteps=schedule,
    )

In [ ]:
# Playground controls
SEED = 42
NUM_STEPS = 20

torch.cuda.synchronize(DEVICE)
torch.cuda.reset_peak_memory_stats(DEVICE)
baseline_allocated = torch.cuda.memory_allocated(DEVICE)
generation_started = time.perf_counter()

generation = generate_from_context(
    model,
    context,
    num_steps=NUM_STEPS,
    seed=SEED,
)

torch.cuda.synchronize(DEVICE)
generation_seconds = time.perf_counter() - generation_started
generation_report = {
    "seed": SEED,
    "steps": NUM_STEPS,
    "generation_and_decode_seconds": generation_seconds,
    "context_shape": tuple(context.shape),
    "latent_shape": tuple(generation.latents.shape),
    "image_shape": tuple(generation.images.shape),
    "allocated_before_sampling_GiB": baseline_allocated / 2**30,
    "peak_allocated_GiB": torch.cuda.max_memory_allocated(DEVICE) / 2**30,
}
print("Generation report:", generation_report)

## 9. Inspect the raw image, quantized canvas, strict decode, and metrics

Clamping is used only to convert the raw VAE tensor into a viewable image. The strict ARC decode has already used the original unclamped tensor and never receives the expected dimensions.

In [ ]:
from IPython.display import display
from PIL import Image

from diffusion_arc.grid_codec import ARC_PALETTE_RGB, classify_arc_cells


def raw_image_to_pil(image: torch.Tensor) -> Image.Image:
    rgb = (
        image.detach()
        .float()
        .cpu()
        .clamp(-1, 1)
        .add(1)
        .mul(127.5)
        .round()
        .to(torch.uint8)
        .permute(1, 2, 0)
        .numpy()
    )
    return Image.fromarray(rgb, mode="RGB")


def arc_grid_to_pil(grid, *, scale: int = 12) -> Image.Image:
    classes = torch.as_tensor(grid, dtype=torch.int64).cpu()
    if classes.ndim != 2 or classes.numel() == 0:
        raise ValueError("grid must be a non-empty two-dimensional array.")
    if int(classes.min()) < 0 or int(classes.max()) > 10:
        raise ValueError("grid contains a class outside the ARC/PAD palette.")
    rgb = ARC_PALETTE_RGB[classes]
    rgb = rgb.repeat_interleave(scale, dim=0).repeat_interleave(scale, dim=1)
    return Image.fromarray(rgb.numpy(), mode="RGB")


def strict_metrics(decode, expected_grid):
    result = {
        "valid": decode.valid,
        "reason": decode.reason,
        "predicted_height": decode.height,
        "predicted_width": decode.width,
        "shape_match": None if expected_grid is None else False,
        "cell_accuracy": None,
        "exact_grid": None if expected_grid is None else False,
    }
    if expected_grid is None or not decode.valid:
        return result
    expected = torch.as_tensor(expected_grid, dtype=torch.int64).cpu()
    predicted = decode.grid.detach().to(dtype=torch.int64, device="cpu")
    result["shape_match"] = tuple(predicted.shape) == tuple(expected.shape)
    if result["shape_match"]:
        matches = predicted == expected
        result["cell_accuracy"] = float(matches.float().mean().item())
        result["exact_grid"] = bool(matches.all().item())
    return result

In [ ]:
raw_image = generation.images[0]
quantized_canvas = classify_arc_cells(raw_image).detach().cpu()
decode = generation.decodes[0]
metrics = strict_metrics(decode, EXPECTED_GRID)

print("Raw VAE image [3, 480, 480]:")
display(raw_image_to_pil(raw_image))

print("Nearest-palette 30×30 canvas, including white PAD class:")
display(arc_grid_to_pil(quantized_canvas))

print("Strict decode:", metrics)
if decode.valid:
    predicted_grid = decode.grid.detach().cpu()
    print("Predicted grid:")
    for row in predicted_grid.tolist():
        print(" ".join(str(value) for value in row))
    display(arc_grid_to_pil(predicted_grid, scale=32))
else:
    print("No grid is returned because the generated PAD boundary is invalid.")

if EXPECTED_GRID is not None:
    print("Expected grid (evaluation only):")
    display(arc_grid_to_pil(EXPECTED_GRID, scale=32))

## How to experiment

- Change only SEED or NUM_STEPS, then rerun the generation and display cells. The cached NVARC context is reused.
- Change TASK or TEST_INDEX, then rerun from the task cell through the end so the prompt and context are rebuilt.
- Keep EXPECTED_GRID separate or set it to None. It is never used to infer output dimensions.
- Long ARC prompts increase dense attention over the prompt tokens plus 900 image tokens and can raise memory use sharply.
- If CUDA runs out of memory, reconnect the runtime and start with the small task again. The notebook deliberately has no silent CPU, dtype, or model fallback.
- A valid run proves that shapes, devices, dtypes, hidden-state extraction, FLUX sampling, VAE decoding, and strict ARC parsing connect. It does not prove that untrained NVARC conditioning is meaningful to FLUX.